# Sheet 05 - Variational Autoencoders

Introduction to Deep Learning - Summer Semester 2026

Ulf Krumnack & Robin Rawiel - Universität OsnabrückDue: May 24, 2026

In this exercise sheet you will focus on the VAE part of generative
learning from lecture 07. The goal is to connect the probabilistic ideas
from the lecture and script to a compact MNIST implementation. You will
derive the ELBO ingredients, implement a VAE with the reparameterization
trick, and inspect how reconstruction quality and latent-space
regularization interact in practice.

## Task 1: Theory and VAE Intuition \[6 points\]

### 1.1 Latent Variable Models and the ELBO \[2 points\]

1.  What is a **latent variable model**? Write down the marginal
    likelihood of an observation $x$ in terms of a latent variable $z$.

2.  Why is the marginal likelihood typically intractable for deep
    generative models, and how does the **Evidence Lower Bound (ELBO)**
    $\log p(x) \geq \mathbb{E}_{q_\phi(z\mid x)}[\log p_\theta(x\mid z)] - D_{\mathrm{KL}}\bigl(q_\phi(z\mid x) \| p(z)\bigr)$
    address this? Explain the role of the reconstruction term and the KL
    term.

### 1.2 Reparameterization and Stochastic Layers \[2 points\]

1.  Why can we not directly backpropagate through a naive sample
    $z \sim \mathcal{N}(\mu, \sigma^2)$ inside a VAE, and how does the
    **reparameterization trick** $z = \mu + \sigma \odot \epsilon$ with
    $\epsilon \sim \mathcal{N}(0, I)$ make gradient-based training
    possible?
    
    Answer: You cant really do that, how do you take derivate from random result. By changing input of random procedure you end up having random result. So the idea is to put a leash on randomness and introduce the noise. The reparameterization
    trick fixes this by sampling $\epsilon \sim \mathcal{N}(0, I)$ externally as a
    fixed constant, then computing $z = \mu + \sigma \times \epsilon$
    deterministically.

2.  Why does the encoder output a mean and a log-variance instead of a
    single deterministic latent vector?

    Answer: A standard autoencoder maps each image to one fixed point z in latent
    space. This creates vast empty regions between training points — if you
    sample from those regions, the decoder produces garbage. By outputting
    a distribution (mean μ and variance σ²), the encoder expresses
    uncertainty about the exact latent location. This forces overlapping
    blobs across the latent space, filling the holes and making random
    sampling reliable for generation.


### 1.3 Reconstruction, Regularization, and Blurriness \[2 points\]

The lecture and script discuss blurriness mainly for Gaussian decoders
and squared-error reconstruction. The important idea for this task is
the same more generally: pixel-wise losses encourage averaging when the
model is uncertain about fine detail.

1.  What does the KL term encourage in the latent space of a VAE, and
    why can a standard VAE produce blurry reconstructions or samples
    when the decoder is trained with a pixel-wise reconstruction term?

2.  What is the main idea of a **VQ-VAE**, and why can discrete latent
    codes help reduce blur?

## Common Setup

The next two code cells provide the shared setup used in the
generative-model experiments below. To keep the runtime manageable on
your laptops, we work on subsets of MNIST rather than the full dataset.

In this part you should:

- load the MNIST subsets and create the data loaders,
- complete the two helper functions `scale_to_tanh` and `show_grid`, and
- run the second cell to print the dataset shapes and display a few
  example digits.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.ToTensor()
full_train_dataset = torchvision.datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
full_test_dataset = torchvision.datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

train_subset = Subset(full_train_dataset, range(8000))
test_subset = Subset(full_test_dataset, range(1000))

train_loader = DataLoader(train_subset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)


def scale_to_tanh(images):
    """Map images from [0, 1] to [-1, 1] for the GAN."""
    # TODO: Your solution here


def show_grid(images, title, nrow=8, normalize=False, value_range=None):
    """Utility function to display image batches."""
    # TODO: Your solution here

100%|██████████| 9.91M/9.91M [00:01<00:00, 9.46MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 256kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 2.38MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.12MB/s]


In [ ]:
# TODO: Your solution here

## Task 2: VAE on MNIST \[10 points\]

**Learning objectives:**

- Implement a VAE with the reparameterization trick and ELBO loss
- Train it on MNIST and inspect reconstructions, prior samples, and
  interpolations
- Visualize the learned latent space on the test set

Tip: get the VAE working first and make sure the loss decreases before
you move on to the visualizations. The later subtasks reuse the same
trained model and are mainly there to help you interpret what the VAE
has learned.

### 2.1 VAE Architecture \[2 points\]

Implement a VAE with a fully connected encoder and decoder. The encoder
should predict both the latent mean $\mu$ and the log-variance
$\log \sigma^2$.

In particular, complete the following parts:

- define the shared encoder network,
- add one linear layer for `mu` and one for `logvar`,
- implement the reparameterization step, and
- complete `decode` and `forward` so that the model returns
  `(x_hat, mu, logvar)`.

After that, instantiate the model and print it once to check that the
architecture looks sensible.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=256, latent_dim=16):
        super().__init__()
        self.latent_dim = latent_dim
        # TODO: Your solution here

    def encode(self, x):
        # TODO: Your solution here

    def reparameterize(self, mu, logvar):
        """Sample z = mu + std * eps with eps ~ N(0, I)."""
        # TODO: Your solution here

    def decode(self, z):
        # TODO: Your solution here

    def forward(self, x):
        # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 2.2 ELBO Loss on MNIST \[2 points\]

Implement the ELBO loss for this MNIST setup.

In the lecture and script, the reconstruction term was motivated through
a Gaussian decoder, which leads to squared error. Here we instead use a
Bernoulli-style decoder with a sigmoid output and binary cross-entropy,
because MNIST pixels are normalized to $[0, 1]$ and are close to binary.
The ELBO structure stays the same; only the observation model changes.

Your function should return three values:

- the total ELBO loss used for training,
- the reconstruction term, and
- the KL term.

You will use these three values again in the next task to plot the
training curves.

In [ ]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """Return total loss, reconstruction loss, and KL divergence."""
    # TODO: Your solution here

### 2.3 Train and Inspect the VAE \[3 points\]

First complete the training function `train_vae`. Then use it to train
the model and inspect what it has learned.

Your final result in this task should include all of the following:

- a plot of the total, reconstruction, and KL losses over epochs,
- a reconstruction plot comparing original and reconstructed test
  images,
- a grid of samples drawn from the prior, and
- one latent interpolation between two test digits.

When you run the plotting cell, check whether the reconstructions are
recognisable and whether the interpolation changes smoothly.

In [ ]:
def train_vae(model, loader, epochs=10, lr=1e-3, beta=1.0):
    # TODO: Your solution here

In [ ]:
# TODO: Your solution here

### 2.4 Visualize the Latent Space \[1 point\]

Encode the test set, keep the latent means `mu`, and make a scatter plot
of the first two latent coordinates colored by digit label.

Because the latent dimension is 16, this is only a two-dimensional
projection. Still, it is a useful sanity check for whether nearby digits
occupy nearby regions in latent space.

In [ ]:
# TODO: Your solution here

### 2.5 Interpret the VAE Results \[2 points\]

Answer the following questions after inspecting your reconstructions,
prior samples, interpolation, and latent scatter plot:

1.  Which plots indicate that the latent space is structured well enough
    for interpolation and sampling?

2.  Why is binary cross-entropy a reasonable reconstruction loss for
    this MNIST experiment, even though the lecture and script often
    discuss squared error, and what qualitative change would you expect
    if $\beta$ in the ELBO were increased strongly?

## Task 3: The $\beta$-VAE Trade-off \[4 points\]

**Learning objectives:**

- Observe the trade-off between reconstruction quality and latent-space
  regularization empirically
- Connect the effect of $\beta$ to the lecture discussion of structured
  latent spaces and blurry samples

For this task, you do not need any additional theory beyond the ELBO
from above. Here, $\beta$ simply rescales the KL term in
$\text{total loss} = \text{reconstruction loss} + \beta \cdot \text{KL loss}$.
Larger $\beta$ means stronger pressure toward the prior, while smaller
$\beta$ lets the model spend more capacity on reconstruction.

### 3.1 Short $\beta$-VAE Experiment \[2 points\]

Train two additional VAEs with clearly different KL weights, for example
$\beta = 0.01$ and $\beta = 8.0$.

In this task you should:

- train both models for a short run,
- plot the reconstruction and KL curves for both settings, and
- compare reconstructions of the same test digits and one grid of prior
  samples from each model.

Use only a few epochs so that the experiment stays lightweight. Reuse
the same `VAE` class and the same `train_vae` function from Task 2. The
only change is the value of `beta` in the loss.

In [ ]:
def sample_from_model(model, n_samples=64):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n_samples, model.latent_dim, device=device)
        return model.decode(z).view(-1, 1, 28, 28).cpu()


# TODO: Your solution here

### 3.2 Reflect on the Effect of $\beta$ \[2 points\]

Answer the following questions after running the experiment:

1.  Which $\beta$ value produced better reconstructions in your short
    run, and which one pushed the latent distribution more strongly
    toward the prior? Explain how you can see this in the reconstruction
    plots and the KL curves.

2.  How does this experiment connect to the lecture discussion of
    latent-space structure, reconstruction quality, and blur?